# 03 — Guesser training orchestration

This notebook runs the existing A/B training and comparison scripts; it does not reimplement their model or training loop. It calls the Python 3.13 model project with `uv run` because this notebook kernel belongs to the separate Python 3.12 evaluation project. Do not import the training modules directly into this kernel.

Training is opt-in and disabled by default. This notebook uses data already prepared by `01_dataset_pipeline.ipynb` and checked by `02_dataset_audit.ipynb`; it never downloads or prepares data. Training reads only the selected profile’s train and validation splits. The default profile is `small`. Full training can take hours and use the GPU.

## Launch and development workflow

From the repository root, sync the model environment if it has not already been installed, then launch JupyterLab through the evaluation environment with the repository as its contents root. Do not append the notebooks directory as a positional argument; Jupyter then uses that directory as the server root. Open `models/guesser/evaluation/notebooks` in the file browser:

```sh
uv sync --project models/guesser --frozen
uv run --project models/guesser/evaluation jupyter lab --ServerApp.root_dir=.
```

The model scripts are the source of truth. The notebook invokes `training.train` and `training.compare_ab` as checked subprocesses from `models/guesser/`; it does not copy or import their implementation into the notebook kernel. The subprocess environment removes inherited `PYTHONPATH` and `PYTHONHOME` so unrelated Python installations cannot shadow the model environment. No shell-specific activation is needed.

> **Expected cross-project warning:** `uv` may report that the evaluation `VIRTUAL_ENV` does not match `models/guesser/.venv`. This is intentional: the notebook kernel uses Python 3.12, while the model scripts use the separate Python 3.13 model environment. `--project` deliberately selects the model environment; do not add `--active`, which would force the notebook environment. This exact mismatch warning is harmless by itself; other errors or nonzero command results still indicate a problem.

Use the options cell to select `small`, `medium`, or `production`, fast/full mode, worker count, and checkpoint directory. The status and preflight cells are read-only. Set `RUN_TRAINING = True` only when the prepared data and GPU are ready. Compare only after both A and B have completed under the same checkpoint root.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from guesser_evaluation import notebook_tools

# Find the repository from the installed evaluation package, not a fixed home path.
NOTEBOOK_TOOLS = notebook_tools.module_source_path("notebook_tools.py")
REPOSITORY_ROOT = NOTEBOOK_TOOLS.parents[5]
MODEL_PROJECT = REPOSITORY_ROOT / "models" / "guesser"
if not (MODEL_PROJECT / "training" / "train.py").is_file():
    raise FileNotFoundError(f"Model training scripts not found under {MODEL_PROJECT}")

print("Repository:", REPOSITORY_ROOT)
print("Model project:", MODEL_PROJECT)

## Options and prepared data

`PREPARED_DIR` must contain prepared train and validation manifests. Use the same dataset profile and checkpoint root for both A and B. Fast mode is a short pipeline check (one epoch, two train batches and one validation batch per arm); it is not a performance result. Full mode uses the script defaults unless overridden here. Worker counts are machine-dependent; start conservatively and tune from observed CPU/RAM use.

The fast and full modes use separate checkpoint roots by default so a short diagnostic cannot be mistaken for a full experiment.

In [ ]:
# Keep execution disabled until the options and resource cost have been reviewed.
RUN_TRAINING = True
TRAINING_MODE = "full"  # 'fast' or 'full'
DATA_PROFILE = "production"  # 'small', 'medium', or 'production'
RUN_SELECTION = "both"  # 'a', 'b', or 'both'
NUM_WORKERS = 10
MAX_EPOCHS = 6
BATCH_SIZE = 8
ACCUMULATE_GRAD_BATCHES = 4
RESUME = True
RUN_COMPARISON = True

if DATA_PROFILE not in {"small", "medium", "production"}:
    raise ValueError("Choose small, medium, or production data.")
if TRAINING_MODE not in {"fast", "full"}:
    raise ValueError("TRAINING_MODE must be fast or full.")
if RUN_SELECTION not in {"a", "b", "both"}:
    raise ValueError("RUN_SELECTION must be a, b, or both.")
if NUM_WORKERS < 0:
    raise ValueError("NUM_WORKERS must be non-negative.")

PREPARED_DIR = MODEL_PROJECT / "data" / ".local" / "data" / DATA_PROFILE / "prepared"
FAST_CHECKPOINT_ROOT = MODEL_PROJECT / "checkpoints" / f"notebook-{DATA_PROFILE}-fast"
FULL_CHECKPOINT_ROOT = MODEL_PROJECT / "checkpoints" / f"notebook-{DATA_PROFILE}-full"
CHECKPOINT_ROOT = FAST_CHECKPOINT_ROOT if TRAINING_MODE == "fast" else FULL_CHECKPOINT_ROOT

print("Training enabled:", RUN_TRAINING)
print("Mode / profile / arms:", TRAINING_MODE, DATA_PROFILE, RUN_SELECTION)
print("Prepared data:", PREPARED_DIR)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Workers:", NUM_WORKERS)

## Readiness check

This checks the prepared-data summary/manifests and imports the model stack inside the model project’s own locked environment. It does not sync packages, download Quick, Draw! data, or start training. If the model environment is missing dependencies, the error names the one-time sync command. A prepared dataset summary is checked against the selected profile before training.

In [ ]:
import json
import os
import subprocess

# Remove host-injected Python paths while keeping other notebook environment settings.
MODEL_ENV = dict(os.environ)
MODEL_ENV.pop("PYTHONPATH", None)
MODEL_ENV.pop("PYTHONHOME", None)
MODEL_ENV["PYTHONUNBUFFERED"] = "1"

SUMMARY_PATH = PREPARED_DIR.parent / "dataset-summary.json"
REQUIRED_MANIFESTS = [
    PREPARED_DIR / "manifests" / f"{split}.parquet" for split in ("train", "validation")
]
missing = [str(path) for path in [SUMMARY_PATH, *REQUIRED_MANIFESTS] if not path.is_file()]
if missing:
    raise FileNotFoundError("Prepared data is incomplete; missing: " + ", ".join(missing))

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
if summary.get("profile") != DATA_PROFILE:
    raise ValueError(
        f"Prepared profile is {summary.get('profile')!r}, not selected {DATA_PROFILE!r}."
    )

# Probe first: --no-sync is deliberate so notebook execution cannot mutate the model environment.
probe_code = (
    "import sys, torch, torchvision, lightning, timm, torchmetrics, safetensors, PIL; "
    'print("python=" + sys.version.split()[0]); '
    'print("executable=" + sys.executable); '
    'print("torch=" + torch.__version__); '
    'print("cuda_available=" + str(torch.cuda.is_available())); '
    'print("pillow=" + PIL.__file__)'
)
probe = subprocess.run(
    [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-c",
        probe_code,
    ],
    cwd=REPOSITORY_ROOT,
    env=MODEL_ENV,
    capture_output=True,
    check=False,
    text=True,
)
if probe.returncode != 0:
    raise RuntimeError(
        "Model environment is not ready. From the repository root, run "
        "`uv sync --project models/guesser --frozen` while no training job is active.\n"
        + (probe.stderr.strip() or probe.stdout.strip())
    )
print(probe.stdout)
print("Prepared profile:", summary["profile"])
print("Train/validation manifests:", *REQUIRED_MANIFESTS, sep="\n  ")

## Run the existing trainer (explicit opt-in)

When enabled, the cell calls `training.train` with explicit profile path, checkpoint root, mode, run selection, worker count and batch settings. Fast and full runs are isolated. It streams the trainer’s output and raises on a failed process, so later cells cannot mistake a failed run for success. Interrupt the cell to request a graceful stop; the runner forwards the signal and waits up to 120 seconds for Lightning’s shutdown, then escalates if needed. Do not run another training job against the same checkpoint root at the same time.

In [ ]:
if not RUN_TRAINING:
    print("Training disabled. Review the options, then set RUN_TRAINING = True to launch.")
else:
    # Match the CLI defaults explicitly so the experiment settings are visible in this notebook.
    training_command = [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-m",
        "training.train",
        "--run",
        RUN_SELECTION,
        "--prepared-dir",
        str(PREPARED_DIR),
        "--checkpoint-root",
        str(CHECKPOINT_ROOT),
        "--num-workers",
        str(NUM_WORKERS),
        "--max-epochs",
        str(MAX_EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--accumulate-grad-batches",
        str(ACCUMULATE_GRAD_BATCHES),
        "--seed",
        "198173",
        "--precision",
        "32-true",
    ]
    if TRAINING_MODE == "fast":
        training_command.append("--fast")
    if RESUME:
        training_command.append("--resume")
    print("Launching:", " ".join(training_command), flush=True)
    if notebook_tools.run_interruptible_command(training_command, cwd=MODEL_PROJECT, env=MODEL_ENV):
        print("Training command completed successfully.")

## Compare completed A/B runs

The existing comparison script reads training/validation logs and writes a Markdown/JSON summary plus SVG plots. It is not the fixed ten-stage held-out evaluation. Enable comparison only after both A and B have finished in the selected checkpoint root; it does not open test data. For full runs, use the full-mode checkpoint root.

In [ ]:
if not RUN_COMPARISON:
    print("Comparison disabled. Set RUN_COMPARISON = True after both runs finish.")
else:
    if RUN_SELECTION != "both":
        raise ValueError('A/B comparison requires RUN_SELECTION = "both".')
    if TRAINING_MODE != "full":
        raise ValueError(
            "Use full mode for the documented A/B comparison; fast runs are pipeline checks only."
        )
    run_directories = [
        CHECKPOINT_ROOT / "A_complete_only" / "logs",
        CHECKPOINT_ROOT / "B_partial" / "logs",
    ]
    missing_logs = [
        str(path)
        for path in run_directories
        if not path.is_dir() or not list(path.rglob("metrics.csv"))
    ]
    if missing_logs:
        raise FileNotFoundError(
            "Both full runs need metrics.csv; missing logs under: " + ", ".join(missing_logs)
        )
    comparison_command = [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-m",
        "training.compare_ab",
        "--checkpoint-root",
        str(CHECKPOINT_ROOT),
    ]
    subprocess.run(comparison_command, cwd=MODEL_PROJECT, env=MODEL_ENV, check=True)
    print("Comparison artifacts:", CHECKPOINT_ROOT / "ab_comparison")

## Reproduce the current A/B result

Use the same prepared profile and full checkpoint root for both arms. To reproduce training, set `TRAINING_MODE = 'full'`, `RUN_SELECTION = 'both'`, and `RUN_TRAINING = True`, then run the training cell. After both arms finish, set `RUN_COMPARISON = True` and run the comparison cell. It writes `comparison.md`, `comparison-summary.json`, validation-curve plots, and per-run plots under `<checkpoint-root>/ab_comparison/`. If the runs already exist, skip training and run only comparison against that same root.

This reproduces the code's **training/validation A/B comparison only**. Fast-mode values are smoke diagnostics, not results. The fixed ten-stage recognition evaluator runs separately in `06_ten_stage_evaluation.ipynb`; `04_model_evaluation.ipynb` only checks for the required artifacts. Keep test data out of model selection and threshold tuning.

The authoritative training defaults and behavior remain in [`../training/README.md`](../../training/README.md) and the collaborators’ `training/` package. This notebook is only a controlled, cross-environment launcher.